In [93]:
from pathlib import Path
import json

BASE_PATH = Path.cwd().resolve()
INPUT_PATH = BASE_PATH.joinpath("llm_responses_agentic_rag_results_with_metrics_FINAL.json")
METADATA_BASE_PATH = BASE_PATH.joinpath("chat_data")
OUTPUT_PATH = BASE_PATH.joinpath("llm_responses_agentic_rag_results_with_metrics_and_metadata_FINAL.json")

with open(INPUT_PATH, mode="r", encoding="utf-8") as f:
    data = json.load(f)

model_dirs = list(METADATA_BASE_PATH.glob("*"))

metadata = []

LAW_SPECIALIST_AGENT_FILE_NAME = "law_specialist_agent.json"
RETRIEVAl_AGENT_FILE_NAME = "retrieval_agent.json"
RETRIEVAL_DATA_DIR_NAME = "retrieval_history"

In [94]:
for model_dir in model_dirs:
    question_dirs = model_dir.glob("*")
    for question_dir in question_dirs:
        rag_data_used = []
        pregunta, subpregunta = question_dir.name.split("_")
        with open(question_dir.joinpath(LAW_SPECIALIST_AGENT_FILE_NAME), mode="r") as f:
            law_specialist_agent_data = json.load(f)
        with open(question_dir.joinpath(RETRIEVAl_AGENT_FILE_NAME), mode="r") as f:
            retrieval_agent_data = json.load(f)
        for retrieval_file_path in question_dir.joinpath(RETRIEVAL_DATA_DIR_NAME).glob("*"):
            with open(retrieval_file_path, mode="r") as f:
                rag_data_used.append(json.load(f))
            
        metadata_item = {
            "law_specialist_model": model_dir.name.replace("_", ":"),
            "numero_pregunta": int(pregunta),
            "numero_subpregunta": int(subpregunta) if subpregunta != "00" else None,
            "law_specialist_agent": law_specialist_agent_data["metadata"],
            "retrieval_agent": retrieval_agent_data["metadata"],
            "rag_data_used": rag_data_used
        }
        metadata.append(metadata_item)

for item in data:
    pregunta_subpregunta = (item["numero_pregunta"], item["numero_subpregunta"])
    for model_response in item["llm_responses_agentic_rag"]:
        model_response["metadata"] = next(filter(lambda x: ((x["numero_pregunta"], x["numero_subpregunta"]) == pregunta_subpregunta) and x["law_specialist_model"] == model_response["law_specialist_model"], metadata))

with open(OUTPUT_PATH, mode="w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)